# ROGII Sidecar Model Package Inference


In [ ]:
from pathlib import Path

# Attach a Kaggle Dataset containing a hidden-safe model package.
MODEL_PACKAGE_ROOTS = [
    Path("/kaggle/input/datasets/pilkwang/rogii-model-package"),
    Path("/kaggle/input/rogii-model-package"),
    Path("/kaggle/input/rogii-branch-artifacts/rogii_model_package"),
]

# Competition data root. Leave as auto unless the dataset slug changes.
COMPETITION_DATA_ROOTS = [
    Path("/kaggle/input/rogii-wellbore-geology-prediction"),
    Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction"),
]

STRICT_MODEL_PACKAGE = True
OUTPUT_SUBMISSION_NAME = "submission.csv"
WRITE_DEBUG_REPORTS = True

# Optional TVT clipping. Keep None unless a package manifest explicitly recommends bounds.
TVT_CLIP_MIN = None
TVT_CLIP_MAX = None


## Inference Contract

The package must include:

```text
metadata/model_package_manifest.json
feature_builders/build_features.py
feature_builders/feature_columns.json
stacking/blend_config.json        # or the path named by manifest["blend_config"]
models/...
```

`build_features.py` should expose `build_features(...)` and return one row per sample id, including model feature columns and `last_known_TVT` when predictions are residual deltas.

Runtime rules kept intentionally strict:

- Public `preds/*_test*` files are not used.
- Feature builder imports may use helper modules inside `feature_builders/`.
- All blend members must use the same `target_space` for now.
- NaN handling is native by default; use `fillna` or `missing_value_policy` in the manifest only when the model needs it.
- `blend_config.postprocess` may apply `alpha`, `fade_tau_md`, and optional well-level Savitzky-Golay smoothing.

The manifest may also include `model_type="direct_feature"` entries. Those do not load a model file; they copy a feature column such as a direct plane-formula residual into the blend as another prediction member.


In [ ]:
import importlib.util
import inspect
import json
import pickle
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display


def _existing_path(paths):
    for path in paths:
        if Path(path).exists():
            return Path(path)
    return None


def find_competition_root() -> Path:
    root = _existing_path(COMPETITION_DATA_ROOTS)
    if root is not None and (root / "sample_submission.csv").exists() and (root / "test").exists():
        return root
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for sample in input_root.glob("**/sample_submission.csv"):
            candidate = sample.parent
            if (candidate / "test").exists():
                return candidate
    raise RuntimeError("Could not find competition sample_submission.csv and test/ under /kaggle/input.")


def find_model_package_root() -> Path:
    for root in MODEL_PACKAGE_ROOTS:
        root = Path(root)
        if (root / "metadata" / "model_package_manifest.json").exists():
            return root
    for manifest in Path("/kaggle/input").glob("**/metadata/model_package_manifest.json"):
        return manifest.parents[1]
    raise RuntimeError(
        "No hidden-safe model package found. Static public prediction artifacts are not valid for hidden reruns. "
        "Attach a Dataset containing metadata/model_package_manifest.json, models/, feature_builders/, and stacking/."
    )


def read_json(path: Path) -> dict[str, Any]:
    with path.open() as f:
        return json.load(f)


def manifest_path(manifest: dict[str, Any], key: str, default: str) -> str:
    value = manifest.get(key, default)
    if isinstance(value, str) and value.strip():
        return value
    raise RuntimeError(f"Manifest field {key!r} must be a relative file path string.")


def prediction_column_for_entry(entry: dict[str, Any]) -> str:
    if entry.get("prediction_column"):
        return str(entry["prediction_column"])
    branch_name = entry.get("branch_name")
    model_name = entry.get("model_name")
    if not branch_name or not model_name:
        raise RuntimeError(f"Model entry needs prediction_column or branch_name/model_name: {entry}")
    return f"pred_delta_{branch_name}_{model_name}"


def validate_manifest(package_root: Path, manifest: dict[str, Any]) -> None:
    required = ["schema_version", "package_type", "hidden_inference_supported", "feature_sets", "models", "blend_config"]
    missing = [field for field in required if field not in manifest]
    if missing:
        raise RuntimeError(f"Model package manifest is missing fields: {missing}")
    if manifest.get("hidden_inference_supported") is not True:
        raise RuntimeError("Model package manifest does not set hidden_inference_supported=true.")
    if manifest.get("package_type") not in {"rogii_hidden_model_package", "hidden_model_package"}:
        raise RuntimeError(f"Unexpected model package type: {manifest.get('package_type')!r}")
    if not isinstance(manifest.get("models"), list) or not manifest.get("models"):
        raise RuntimeError("Manifest must contain a non-empty models list.")

    feature_columns_rel = manifest_path(manifest, "feature_columns", "feature_builders/feature_columns.json")
    blend_config_rel = manifest_path(manifest, "blend_config", "stacking/blend_config.json")
    for rel in [feature_columns_rel, blend_config_rel]:
        if not (package_root / rel).exists():
            raise RuntimeError(f"Model package is missing {rel}")

    blend_config = read_json(package_root / blend_config_rel)
    blend_space = blend_config.get("target_space") or blend_config.get("prediction_space") or manifest.get("target_space", "delta")
    if blend_space not in {"delta", "tvt"}:
        raise RuntimeError(f"Unsupported blend target_space={blend_space!r}; expected 'delta' or 'tvt'.")

    allowed_model_types = {
        "lightgbm_booster",
        "lightgbm_sklearn_pickle",
        "xgboost_json",
        "xgboost_pickle",
        "catboost_cbm",
        "sklearn_pickle",
        "direct_feature",
    }
    prediction_columns = set()
    for idx, entry in enumerate(manifest.get("models", [])):
        model_type = entry.get("model_type")
        if model_type not in allowed_model_types:
            raise RuntimeError(f"Unsupported model_type in manifest entry {idx}: {model_type!r}")
        pred_col = prediction_column_for_entry(entry)
        if pred_col in prediction_columns:
            raise RuntimeError(f"Missing or duplicated prediction_column in model entry {idx}: {entry}")
        prediction_columns.add(pred_col)
        entry_space = entry.get("target_space", blend_space)
        if entry_space != blend_space:
            raise RuntimeError(
                f"Mixed target_space is not supported: {pred_col} has {entry_space!r}, blend uses {blend_space!r}."
            )
        if model_type == "direct_feature":
            if not entry.get("feature_column"):
                raise RuntimeError(f"direct_feature entry must define feature_column: {entry}")
            continue
        rel = entry.get("path")
        if not rel or not (package_root / rel).exists():
            raise RuntimeError(f"Model package is missing model file for entry: {entry}")


def validate_submission_ids(df: pd.DataFrame, sample: pd.DataFrame, label: str) -> pd.DataFrame:
    if not {"id", "tvt"}.issubset(df.columns):
        raise RuntimeError(f"{label}: expected columns ['id', 'tvt']; got {list(df.columns)}")
    frame = df[["id", "tvt"]].copy()
    frame["id"] = frame["id"].astype(str)
    sample_ids_frame = sample[["id"]].copy()
    sample_ids_frame["id"] = sample_ids_frame["id"].astype(str)
    if frame["id"].duplicated().any():
        dup = frame.loc[frame["id"].duplicated(), "id"].head(10).tolist()
        raise RuntimeError(f"{label}: duplicate ids: {dup}")
    missing = sorted(set(sample_ids_frame["id"]) - set(frame["id"]))
    extra = sorted(set(frame["id"]) - set(sample_ids_frame["id"]))
    if missing:
        raise RuntimeError(f"{label}: missing {len(missing)} sample ids; examples={missing[:10]}")
    if extra:
        raise RuntimeError(f"{label}: extra {len(extra)} ids; examples={extra[:10]}")
    aligned = sample_ids_frame.merge(frame, on="id", how="left")
    if aligned["tvt"].isna().any():
        bad = aligned.loc[aligned["tvt"].isna(), "id"].head(10).tolist()
        raise RuntimeError(f"{label}: NaN after alignment; examples={bad}")
    if not np.isfinite(aligned["tvt"].to_numpy(dtype=float)).all():
        raise RuntimeError(f"{label}: non-finite tvt values")
    return aligned[["id", "tvt"]]


competition_root = find_competition_root()
package_root = find_model_package_root()
manifest = read_json(package_root / "metadata" / "model_package_manifest.json")
validate_manifest(package_root, manifest)
sample_submission = pd.read_csv(competition_root / "sample_submission.csv")

summary = pd.Series({
    "competition_root": competition_root.as_posix(),
    "package_root": package_root.as_posix(),
    "schema_version": manifest.get("schema_version"),
    "blend_config": manifest_path(manifest, "blend_config", "stacking/blend_config.json"),
    "models": len(manifest.get("models", [])),
    "sample_rows": len(sample_submission),
}).to_frame("value")
display(summary)


In [ ]:
def load_feature_builder(package_root: Path):
    feature_dir = package_root / "feature_builders"
    for import_root in [package_root, feature_dir]:
        key = str(import_root)
        if key not in sys.path:
            sys.path.insert(0, key)

    candidates = [
        feature_dir / "build_features.py",
        feature_dir / "feature_builder.py",
    ]
    for path in candidates:
        if path.exists():
            spec = importlib.util.spec_from_file_location("rogii_sidecar_feature_builder", path)
            if spec is None or spec.loader is None:
                raise RuntimeError(f"Could not import feature builder: {path}")
            module = importlib.util.module_from_spec(spec)
            sys.modules[spec.name] = module
            spec.loader.exec_module(module)
            for fn_name in ["build_features", "build_tail_features", "make_features"]:
                if hasattr(module, fn_name):
                    return getattr(module, fn_name), path
    raise RuntimeError(
        "Model package has no feature_builders/build_features.py. Hidden-safe inference requires rebuilding features "
        "for the current Kaggle test wells."
    )


def call_feature_builder(builder, *, data_dir: Path, sample: pd.DataFrame, package_root: Path, manifest: dict[str, Any]) -> pd.DataFrame:
    possible_kwargs = {
        "data_dir": data_dir,
        "competition_root": data_dir,
        "sample_submission": sample,
        "sample": sample,
        "package_root": package_root,
        "manifest": manifest,
        "config": manifest,
    }
    sig = inspect.signature(builder)
    kwargs = {name: value for name, value in possible_kwargs.items() if name in sig.parameters}
    features = builder(**kwargs)
    if not isinstance(features, pd.DataFrame):
        raise RuntimeError("Feature builder must return a pandas DataFrame.")
    if "id" not in features.columns:
        raise RuntimeError("Feature frame must include an 'id' column.")
    features = features.copy()
    features["id"] = features["id"].astype(str)
    sample_ids = sample[["id"]].copy()
    sample_ids["id"] = sample_ids["id"].astype(str)
    if features["id"].duplicated().any():
        dup = features.loc[features["id"].duplicated(), "id"].head(10).tolist()
        raise RuntimeError(f"Feature frame contains duplicate ids: {dup}")
    missing = sorted(set(sample_ids["id"]) - set(features["id"]))
    extra = sorted(set(features["id"]) - set(sample_ids["id"]))
    if missing or extra:
        raise RuntimeError(f"Feature frame id mismatch: missing={len(missing)}, extra={len(extra)}, missing_examples={missing[:10]}")
    aligned = sample_ids.merge(features, on="id", how="left")
    return aligned


def load_feature_columns(package_root: Path, manifest: dict[str, Any]) -> Any:
    return read_json(package_root / manifest_path(manifest, "feature_columns", "feature_builders/feature_columns.json"))


def feature_columns_for_model(feature_columns: Any, entry: dict[str, Any]) -> list[str]:
    if isinstance(entry.get("feature_columns"), list):
        return list(entry["feature_columns"])
    feature_set = entry.get("feature_set")
    if isinstance(feature_columns, list):
        return list(feature_columns)
    if isinstance(feature_columns, dict):
        if feature_set and isinstance(feature_columns.get(feature_set), list):
            return list(feature_columns[feature_set])
        if isinstance(feature_columns.get("columns"), list):
            return list(feature_columns["columns"])
    raise RuntimeError(f"Could not resolve feature columns for model entry: {entry}")


builder, builder_path = load_feature_builder(package_root)
feature_frame = call_feature_builder(
    builder,
    data_dir=competition_root,
    sample=sample_submission,
    package_root=package_root,
    manifest=manifest,
)
feature_columns_config = load_feature_columns(package_root, manifest)

feature_memory_mb = float(feature_frame.memory_usage(deep=True).sum() / 1024**2)
display(pd.Series({
    "feature_builder": builder_path.as_posix(),
    "feature_rows": len(feature_frame),
    "feature_columns_total": len(feature_frame.columns),
    "feature_memory_mb": round(feature_memory_mb, 2),
}).to_frame("value"))


In [ ]:
def load_model(package_root: Path, entry: dict[str, Any]):
    model_type = entry.get("model_type")
    path = package_root / entry["path"]
    if model_type == "lightgbm_booster":
        import lightgbm as lgb
        return lgb.Booster(model_file=str(path))
    if model_type == "xgboost_json":
        import xgboost as xgb
        booster = xgb.Booster()
        booster.load_model(str(path))
        return booster
    if model_type == "catboost_cbm":
        from catboost import CatBoostRegressor
        model = CatBoostRegressor()
        model.load_model(str(path))
        return model
    if model_type in {"lightgbm_sklearn_pickle", "xgboost_pickle", "sklearn_pickle"}:
        try:
            import joblib
            return joblib.load(path)
        except Exception:
            with path.open("rb") as f:
                return pickle.load(f)
    raise RuntimeError(f"Unsupported model_type={model_type!r} for {entry}")


def _feature_matrix_for_model(frame: pd.DataFrame, columns: list[str], entry: dict[str, Any]) -> pd.DataFrame:
    missing = [c for c in columns if c not in frame.columns]
    if missing:
        raise RuntimeError(f"Feature frame missing {len(missing)} columns; examples={missing[:10]}")
    X_df = frame[columns].replace([np.inf, -np.inf], np.nan)
    fill_value = entry.get("fillna", None)
    policy = str(entry.get("missing_value_policy", manifest.get("missing_value_policy", "native"))).lower()
    if fill_value is not None:
        X_df = X_df.fillna(float(fill_value))
    elif policy in {"native", "none", "null"}:
        pass
    elif policy in {"zero", "fill_zero"}:
        X_df = X_df.fillna(0.0)
    else:
        raise RuntimeError(f"Unsupported missing_value_policy={policy!r} for {entry.get('prediction_column')}")
    return X_df


def predict_model(model, model_type: str, frame: pd.DataFrame, columns: list[str], entry: dict[str, Any]) -> np.ndarray:
    X_df = _feature_matrix_for_model(frame, columns, entry)
    if model_type == "xgboost_json":
        import xgboost as xgb
        pred = model.predict(xgb.DMatrix(X_df.to_numpy(dtype=np.float32)))
    else:
        pred = model.predict(X_df)
    pred = np.asarray(pred, dtype=float)
    if pred.ndim > 1:
        pred = pred.reshape(len(frame), -1)[:, 0]
    if len(pred) != len(frame):
        raise RuntimeError(f"Model prediction length mismatch: got {len(pred)}, expected {len(frame)}")
    if not np.isfinite(pred).all():
        raise RuntimeError(f"Model {entry.get('prediction_column')} produced non-finite predictions.")
    return pred


predictions = pd.DataFrame({"id": feature_frame["id"].to_numpy()})
model_report_rows = []
blend_space = None
for entry in manifest.get("models", []):
    pred_col = prediction_column_for_entry(entry)
    model_type = entry.get("model_type")
    if model_type == "direct_feature":
        source_col = entry.get("feature_column")
        if source_col not in feature_frame.columns:
            raise RuntimeError(f"direct_feature source column is missing: {source_col}")
        pred = pd.to_numeric(feature_frame[source_col], errors="coerce").to_numpy(dtype=float)
        if not np.isfinite(pred).all():
            raise RuntimeError(f"direct_feature {source_col} produced non-finite values.")
        predictions[pred_col] = pred
        model_report_rows.append({
            "prediction_column": pred_col,
            "model_type": model_type,
            "feature_count": 1,
            "source_column": source_col,
            "target_space": entry.get("target_space", "delta"),
            "pred_mean": float(np.nanmean(pred)),
            "pred_std": float(np.nanstd(pred)),
            "pred_min": float(np.nanmin(pred)),
            "pred_max": float(np.nanmax(pred)),
        })
        continue
    columns = feature_columns_for_model(feature_columns_config, entry)
    model = load_model(package_root, entry)
    pred = predict_model(model, model_type, feature_frame, columns, entry)
    predictions[pred_col] = pred
    model_report_rows.append({
        "prediction_column": pred_col,
        "model_type": model_type,
        "feature_count": len(columns),
        "source_column": "",
        "target_space": entry.get("target_space", "delta"),
        "pred_mean": float(np.nanmean(pred)),
        "pred_std": float(np.nanstd(pred)),
        "pred_min": float(np.nanmin(pred)),
        "pred_max": float(np.nanmax(pred)),
    })

model_prediction_report = pd.DataFrame(model_report_rows)
display(model_prediction_report)


In [ ]:
def normalize_weights(blend_config: dict[str, Any]) -> dict[str, float]:
    if isinstance(blend_config.get("weights"), dict):
        return {str(k): float(v) for k, v in blend_config["weights"].items()}
    if isinstance(blend_config.get("model_weights"), dict):
        return {str(k): float(v) for k, v in blend_config["model_weights"].items()}
    weights = {}
    for row in blend_config.get("models", []):
        if "prediction_column" in row and "weight" in row:
            weights[str(row["prediction_column"])] = float(row["weight"])
    if weights:
        return weights
    raise RuntimeError("blend_config.json must contain weights, model_weights, or models[{prediction_column, weight}].")


def validate_weights(weights: dict[str, float], blend_config: dict[str, Any]) -> None:
    if not weights:
        raise RuntimeError("Blend weights are empty.")
    bad_finite = {k: v for k, v in weights.items() if not np.isfinite(v)}
    if bad_finite:
        raise RuntimeError(f"Non-finite blend weights: {bad_finite}")
    if bool(blend_config.get("enforce_nonnegative", True)):
        bad_negative = {k: v for k, v in weights.items() if v < -1e-9}
        if bad_negative:
            raise RuntimeError(f"Negative blend weights are not allowed: {bad_negative}")
    weight_sum = float(sum(weights.values()))
    max_weight_sum = float(blend_config.get("max_weight_sum", 1.05))
    if weight_sum > max_weight_sum:
        raise RuntimeError(f"Blend weight sum too large: {weight_sum:.6f} > {max_weight_sum:.6f}")


def _first_existing_column(frame: pd.DataFrame, names: list[str]) -> str | None:
    for name in names:
        if name in frame.columns:
            return name
    return None


def _apply_delta_postprocess(delta: np.ndarray, blend_config: dict[str, Any], features: pd.DataFrame) -> np.ndarray:
    post = blend_config.get("postprocess", {}) or {}
    out = delta.astype(float).copy()
    tau = post.get("fade_tau_md", post.get("tau", None))
    if tau is not None:
        md_col = _first_existing_column(features, ["md_since_ps", "md_since", "md_delta", "MD_since", "md_from_start"])
        if md_col is None:
            raise RuntimeError("postprocess.fade_tau_md was set, but no md_since column is available in feature_frame.")
        md_since = pd.to_numeric(features[md_col], errors="coerce").to_numpy(dtype=float)
        out *= 1.0 - np.exp(-np.maximum(md_since, 0.0) / float(tau))
    out *= float(post.get("alpha", 1.0))
    return out


def _apply_savgol_if_requested(tvt: np.ndarray, blend_config: dict[str, Any], features: pd.DataFrame) -> np.ndarray:
    post = blend_config.get("postprocess", {}) or {}
    window = int(post.get("savgol_window", 0) or 0)
    if window <= 2:
        return tvt
    if window % 2 == 0:
        window += 1
    poly = int(post.get("savgol_poly", 2) or 2)
    try:
        from scipy.signal import savgol_filter
    except Exception as exc:
        raise RuntimeError(f"Savitzky-Golay smoothing requested but scipy is unavailable: {exc}")
    out = tvt.astype(float).copy()
    group_col = _first_existing_column(features, ["well_id", "well", "WELL"])
    row_col = _first_existing_column(features, ["row_index", "row", "sample_index"])
    tmp = pd.DataFrame({"_pos": np.arange(len(out)), "_tvt": out})
    tmp["_group"] = features[group_col].astype(str).to_numpy() if group_col else features["id"].astype(str).str.rsplit("_", n=1).str[0].to_numpy()
    tmp["_order"] = pd.to_numeric(features[row_col], errors="coerce").to_numpy(dtype=float) if row_col else np.arange(len(out), dtype=float)
    for _, grp in tmp.groupby("_group", sort=False):
        if len(grp) < max(window, poly + 2):
            continue
        order = grp.sort_values("_order")
        w = min(window, len(order) if len(order) % 2 == 1 else len(order) - 1)
        if w < poly + 2 or w <= 2:
            continue
        smoothed = savgol_filter(order["_tvt"].to_numpy(dtype=float), window_length=w, polyorder=min(poly, w - 1), mode="interp")
        out[order["_pos"].to_numpy(dtype=int)] = smoothed
    return out


blend_config = read_json(package_root / manifest_path(manifest, "blend_config", "stacking/blend_config.json"))
weights = normalize_weights(blend_config)
validate_weights(weights, blend_config)
missing_pred_cols = [c for c in weights if c not in predictions.columns]
if missing_pred_cols:
    raise RuntimeError(f"Blend config references missing prediction columns: {missing_pred_cols}")

target_space = blend_config.get("target_space") or blend_config.get("prediction_space") or manifest.get("target_space", "delta")
entry_spaces = {
    prediction_column_for_entry(entry): entry.get("target_space", target_space)
    for entry in manifest.get("models", [])
}
wrong_spaces = {col: entry_spaces.get(col) for col in weights if entry_spaces.get(col, target_space) != target_space}
if wrong_spaces:
    raise RuntimeError(f"Mixed target_space is not supported by this notebook: {wrong_spaces}, blend={target_space!r}")

pred_value = np.zeros(len(predictions), dtype=float)
for col, weight in weights.items():
    pred_value += float(weight) * predictions[col].to_numpy(dtype=float)

if target_space == "delta":
    if "last_known_TVT" not in feature_frame.columns:
        raise RuntimeError("Delta-space blend requires feature_frame['last_known_TVT'].")
    pred_value = _apply_delta_postprocess(pred_value, blend_config, feature_frame)
    tvt = feature_frame["last_known_TVT"].to_numpy(dtype=float) + pred_value
elif target_space == "tvt":
    tvt = pred_value
else:
    raise RuntimeError(f"Unsupported blend target_space={target_space!r}; expected 'delta' or 'tvt'.")

tvt = _apply_savgol_if_requested(tvt, blend_config, feature_frame)

clip_min = TVT_CLIP_MIN if TVT_CLIP_MIN is not None else blend_config.get("tvt_clip_min")
clip_max = TVT_CLIP_MAX if TVT_CLIP_MAX is not None else blend_config.get("tvt_clip_max")
if clip_min is not None or clip_max is not None:
    tvt = np.clip(tvt, -np.inf if clip_min is None else float(clip_min), np.inf if clip_max is None else float(clip_max))

submission = pd.DataFrame({"id": feature_frame["id"].to_numpy(), "tvt": tvt})
submission = validate_submission_ids(submission, sample_submission, label="model_package_submission")
submission.to_csv(OUTPUT_SUBMISSION_NAME, index=False)

if WRITE_DEBUG_REPORTS:
    model_prediction_report.to_csv("model_package_prediction_report.csv", index=False)
    pd.DataFrame([{"prediction_column": k, "weight": v} for k, v in weights.items()]).to_csv("model_package_blend_weights.csv", index=False)
    pd.Series({
        "output_submission_name": OUTPUT_SUBMISSION_NAME,
        "rows": len(submission),
        "target_space": target_space,
        "weight_sum": float(sum(weights.values())),
        "tvt_mean": float(np.mean(submission["tvt"])),
        "tvt_std": float(np.std(submission["tvt"])),
        "postprocess": json.dumps(blend_config.get("postprocess", {}) or {}),
    }).to_csv("model_package_submission_summary.csv")

display(submission.head())
print(f"Wrote {OUTPUT_SUBMISSION_NAME} with {len(submission):,} rows")
